# 09 — Model Versioning + MLflow

**Day 2, Step 9.** Simple versioning first, MLflow second — in that order, as
the Build Notes instruct. Metadata is expanded to what an audit actually needs:
without the git SHA, data checksums and library versions, "which model made this
prediction?" has no answer six months later.

In [1]:
import sys, warnings
sys.path.insert(0, "../src"); sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# The lab imports from the factory. Nothing below reimplements pipeline logic.
from hrai.utils.config import get, raw_path, seed
from hrai.utils.io import load_raw, load_processed
from hrai.utils.logger import setup_logging
setup_logging(fmt="human")
print(f"seed={seed()}  |  datasets: {sorted(get('datasets'))}")

seed=42  |  datasets: ['employee_attrition', 'essential_skills', 'hr_performance_engagement', 'occupation_data', 'software_skills']


In [2]:
from hrai.ml.registry import list_versions, load_model, latest_version
import json

print("versions on disk:", [v["version"] for v in list_versions()])
_, metadata = load_model(latest_version())
print(json.dumps({k: v for k, v in metadata.items()
                  if k not in ("metrics", "feature_columns")}, indent=2)[:1600])

versions on disk: ['v1']


2026-08-28 01:54:58 | INFO  | model loaded


{
  "model_name": "Attrition Prediction Model",
  "version": "v1",
  "algorithm": "logistic_regression + sigmoid calibration",
  "training_date": "2026-08-27T19:54:11.751044+00:00",
  "random_seed": 42,
  "operating_threshold": 0.08,
  "risk_bands": {
    "high": 0.6,
    "medium": 0.3
  },
  "encoded_feature_count": 57,
  "target": "Attrition",
  "trained_on": "employee_attrition (Population A)",
  "data_checksums": {
    "employee_attrition": "a5c31e38bd7fafc9"
  },
  "git_sha": "5eed2142b4fb02c8437e0b3dfeb005c3790cacb1",
  "libraries": {
    "python": "3.11.16",
    "pandas": "2.3.3",
    "numpy": "2.4.6",
    "sklearn": "1.9.0",
    "xgboost": "3.2.0",
    "shap": "0.51.0"
  },
  "artifact": "attrition_pipeline.joblib",
  "artifact_sha256": "61dc62b1e7eaf3843e260d5fdd01c17f37adfc1f5f7d21e7d5f0545a389b522b",
  "calibration_method": "sigmoid",
  "calibration_comparison": [
    {
      "method": "sigmoid",
      "oof_brier": 0.09613,
      "saturated_fraction": 0.0,
      "distinct_va

In [3]:
pd.DataFrame(metadata["metrics"]["cv_comparison"])[
    ["model", "average_precision_mean", "recall_mean", "roc_auc_mean"]].round(4)

,model,average_precision_mean,recall_mean,roc_auc_mean
0,logistic_regression,0.6366,0.7351,0.8323
1,xgboost,0.5633,0.4158,0.7920
2,random_forest,0.5492,0.2386,0.7966


## Calibration was chosen on evidence, not preference

Isotonic scored marginally better on Brier but saturates — a wide band of raw
scores maps to exactly 1.0. That is fatal for the counterfactual engine in
notebook 17, which measures risk as a *difference* after nudging one feature.
Sigmoid was selected because it is smooth.

In [4]:
pd.DataFrame(metadata["calibration_comparison"])

,method,oof_brier,saturated_fraction,distinct_values
0,sigmoid,0.09613,0.0000,1172
1,isotonic,0.09300,0.0357,614


MLflow tracks every run: `mlflow ui --backend-store-uri sqlite:///mlflow.db`